# 03 Validation and the single test run

Run this only after `configs/final_model.json` is committed to Git (the frozen configuration).

In [ ]:
import json, os, sys
from pathlib import Path

# Work from the repository root whether the notebook is opened from notebooks/ or the root
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

cfg = json.load(open("configs/dataset.json"))
OUT = Path("results/data_quality"); OUT.mkdir(parents=True, exist_ok=True)
cfg

## Frozen choice and validation scores

In [ ]:
final = json.load(open("configs/final_model.json"))
pd.Series(final["validation_mae"], name="val_mae").sort_values().round(3)

## Test result (reported once)

In [ ]:
m = pd.read_csv("results/final_test_metrics.csv").iloc[0]
print(f"Model: {m['model']}")
print(f"Test MAE {m['MAE']:.2f} km/h  (95% CI {m['MAE_CI_low']:.2f} to {m['MAE_CI_high']:.2f})")
print(f"Test RMSE {m['RMSE']:.2f} km/h")
print(f"Persistence MAE {m['MAE_persistence']:.2f}  |  hour-of-week MAE {m['MAE_hour_of_week']:.2f}")
print(f"Skill vs persistence: {m['skill_vs_persistence']:.3f}")

## Validation vs test MAE for the selected model and baselines

In [ ]:
vm = final["validation_mae"]
sel = m["model"]
tab = pd.DataFrame({
    "validation MAE": [vm[sel], vm["persistence"], vm["hour_of_week"]],
    "test MAE": [m["MAE"], m["MAE_persistence"], m["MAE_hour_of_week"]],
}, index=[sel, "persistence", "hour_of_week"]).round(3)
tab.plot.bar(figsize=(7, 4), rot=0, title="MAE (km/h): validation vs test")
plt.tight_layout(); plt.savefig("results/val_vs_test_mae.png", dpi=150); plt.show()
tab

## Notes
A large gap between validation and test MAE means the test period differs from validation (season, weather). Describe what you see; do not retune.